In [18]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

SETS =  [
    "ZZxReto", # Train
    "ZZy1", # Train
    "ZZx2",  # Val
    "ZZy2", # Val
    "LSG-1", # Test
    "LSG-2", # Test
    "ZZx1-inv", # Test
    "ZZx1",  # Test
    "ZZx2-inv", # Test
    "semiCirc", # Test
]

In [19]:
results_1l = pd.read_excel("resultados-1l.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l

import re

def fix_column_name(col):
    # Ex: "MSE_ZZxReto_theta" -> "R2_ZZxReto_dtheta"
    match = re.match(r"^MSE_(.+)_([^_]+)$", col)
    if match:
        title, name = match.groups()
        return f"R2_{title}_d{name}"
    return col

results.columns = [fix_column_name(c) for c in results.columns]

In [20]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZxReto_theta,R2_ZZxReto_dtheta,R2_ZZy1_theta,R2_ZZy1_dtheta,...,R2_LSG_2_theta,R2_LSG_2_dtheta,R2_ZZx1_inv_theta,R2_ZZx1_inv_dtheta,R2_ZZx1_theta,R2_ZZx1_dtheta,R2_ZZx2_inv_theta,R2_ZZx2_inv_dtheta,R2_semiCirc_theta,R2_semiCirc_dtheta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed4046,[1],0.3,0.7,0.01,4046,0.404867,0.414760,-2.266710,0.255023,...,-2.243985,0.176547,0.067505,0.315835,0.556536,0.389359,-0.113130,0.179034,NaN,NaN
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed4912,[1],0.3,0.7,0.01,4912,0.531574,0.445708,-2.680994,0.266496,...,-2.407028,0.204723,0.157785,0.359204,0.549249,0.423793,-0.189742,0.182456,NaN,NaN
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed5175,[1],0.3,0.7,0.01,5175,0.362141,0.408191,-2.178630,0.252662,...,-2.325280,0.165853,0.025923,0.302295,0.559068,0.385524,-0.116063,0.179271,NaN,NaN
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed1555,[1],0.3,0.7,0.01,1555,0.531795,0.443835,-2.535055,0.266265,...,-2.070264,0.213272,0.210147,0.370387,0.546735,0.416928,-0.149514,0.180179,NaN,NaN
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed1827,[1],0.3,0.7,0.01,1827,0.511496,0.439903,-2.528586,0.264798,...,-2.209185,0.205238,0.170554,0.358905,0.550948,0.417415,-0.161031,0.184123,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2007,model_arch100_r0.9_Ld0.7_Lp0.3_seed2647,[100],0.7,0.3,0.90,2647,-0.644097,0.563243,-16.746343,0.172644,...,0.170524,0.398977,-1.553063,0.581025,-0.169698,0.414298,-3.764239,0.233789,-17.547591,0.062385
2008,model_arch100_r0.9_Ld0.7_Lp0.3_seed2932,[100],0.7,0.3,0.90,2932,-0.718531,0.515783,-20.611296,0.088560,...,-0.786011,0.394722,-1.745157,0.591634,-0.417592,0.363519,-4.185588,0.385220,-15.686628,0.076522
2009,model_arch100_r0.9_Ld0.7_Lp0.3_seed380,[100],0.7,0.3,0.90,380,-1.062078,0.509280,-16.902464,0.138535,...,-0.716253,0.370801,-2.281333,0.552039,-0.543357,0.360054,-3.794423,0.349476,-13.248207,0.142535
2010,model_arch100_r0.9_Ld0.7_Lp0.3_seed9920,[100],0.7,0.3,0.90,9920,-0.511507,0.566883,-18.655483,0.134478,...,0.189357,0.402507,-1.495714,0.590599,-0.133803,0.409335,-4.938306,0.225768,-20.401291,-0.011936


In [23]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZxReto":  "Train",
    "ZZy1":     "Train",
    "ZZx2":     "Val",
    "ZZy2":     "Val",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx1":     "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 10  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
788,model_arch40_r0.9_Ld0.3_Lp0.7_seed1555,[40],-1.685294,-1.031449,-0.373376,-1.175583
604,model_arch31_r0.01_Ld0.3_Lp0.7_seed1827,[31],-1.531894,-1.145728,-0.434310,-1.176389
587,model_arch30_r0.9_Ld0.3_Lp0.7_seed5175,[30],-1.226060,-1.252766,-0.672978,-1.182792
528,model_arch27_r0.9_Ld0.3_Lp0.7_seed1555,[27],-1.388473,-1.316923,-0.445425,-1.188172
565,model_arch29_r0.9_Ld0.3_Lp0.7_seed4046,[29],-1.460881,-1.221840,-0.474722,-1.190868
709,model_arch36_r0.9_Ld0.3_Lp0.7_seed1827,[36],-1.744339,-1.094671,-0.298478,-1.199658
869,model_arch44_r0.9_Ld0.3_Lp0.7_seed1827,[44],-1.889940,-0.896291,-0.374841,-1.205772
821,model_arch42_r0.01_Ld0.3_Lp0.7_seed4912,[42],-1.768872,-1.072943,-0.326204,-1.211079
705,model_arch36_r0.9_Ld0.3_Lp0.7_seed4046,[36],-1.394746,-1.120403,-0.729507,-1.216834
560,model_arch29_r0.01_Ld0.3_Lp0.7_seed4046,[29],-1.450185,-1.204370,-0.587283,-1.217097



📊 MÉTRICAS COMPLETAS - TOP 10 (theta)


,model,Neurons,R2_ZZxReto_theta,R2_ZZy1_theta,R2_ZZx2_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZx1_theta,R2_ZZx2_inv_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
788,model_arch40_r0.9_Ld0.3_Lp0.7_seed1555,[40],0.667572,-4.038160,-0.632977,-1.429921,0.442922,-2.153066,0.658528,0.119559,-0.934824,NaN,-1.685294,-1.031449,-0.373376,-1.175583
604,model_arch31_r0.01_Ld0.3_Lp0.7_seed1827,[31],0.615449,-3.679236,-0.745364,-1.546093,0.457374,-2.495089,0.575830,0.021873,-0.731537,NaN,-1.531894,-1.145728,-0.434310,-1.176389
587,model_arch30_r0.9_Ld0.3_Lp0.7_seed5175,[30],0.499197,-2.951317,-0.803213,-1.702320,0.445779,-3.222330,0.412208,-0.160454,-0.840093,NaN,-1.226060,-1.252766,-0.672978,-1.182792
528,model_arch27_r0.9_Ld0.3_Lp0.7_seed1555,[27],0.619207,-3.396154,-0.706919,-1.926928,0.517869,-2.635226,0.540227,0.031653,-0.681647,NaN,-1.388473,-1.316923,-0.445425,-1.188172
565,model_arch29_r0.9_Ld0.3_Lp0.7_seed4046,[29],0.610859,-3.532621,-0.725710,-1.717970,0.475467,-2.628516,0.552927,0.015702,-0.789191,NaN,-1.460881,-1.221840,-0.474722,-1.190868
709,model_arch36_r0.9_Ld0.3_Lp0.7_seed1827,[36],0.731280,-4.219959,-0.387804,-1.801539,0.535845,-1.882163,0.679280,0.270900,-1.096249,NaN,-1.744339,-1.094671,-0.298478,-1.199658
869,model_arch44_r0.9_Ld0.3_Lp0.7_seed1827,[44],0.653309,-4.433189,-0.747485,-1.045096,0.335726,-1.988120,0.702362,0.085881,-1.010052,NaN,-1.889940,-0.896291,-0.374841,-1.205772
821,model_arch42_r0.01_Ld0.3_Lp0.7_seed4912,[42],0.734169,-4.271913,-0.405327,-1.740559,0.520020,-1.925519,0.684759,0.269021,-1.179303,NaN,-1.768872,-1.072943,-0.326204,-1.211079
705,model_arch36_r0.9_Ld0.3_Lp0.7_seed4046,[36],0.483992,-3.273484,-0.817637,-1.423168,0.344891,-3.218825,0.440227,-0.166054,-1.047774,NaN,-1.394746,-1.120403,-0.729507,-1.216834
560,model_arch29_r0.01_Ld0.3_Lp0.7_seed4046,[29],0.532052,-3.432422,-0.873249,-1.535492,0.407526,-2.955330,0.484046,-0.112465,-0.760193,NaN,-1.450185,-1.204370,-0.587283,-1.217097


In [22]:
final_table.to_excel("BestModels-1l.xlsx")